In [9]:
# Requirements: numpy, pandas, scikit-learn, scipy, matplotlib
# Optional: scikit-learn-extra (for KMedoids). If absent a small PAM fallback is used.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, OPTICS, AgglomerativeClustering
from sklearn.metrics import (rand_score, adjusted_rand_score,
                             mutual_info_score, adjusted_mutual_info_score,
                             normalized_mutual_info_score, silhouette_score,
                             calinski_harabasz_score, davies_bouldin_score)
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings("ignore")

# Try import KMedoids from sklearn_extra
_has_kmedoids = False
try:
    from sklearn_extra.cluster import KMedoids
    _has_kmedoids = True
except Exception:
    _has_kmedoids = False

# Utilities

def relabel_classes(y):
    uniques, inv = np.unique(y, return_inverse=True)
    return inv

def best_cluster_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # convert labels to contiguous ints
    true_labels = np.unique(y_true)
    pred_labels = np.unique(y_pred)
    n_true = true_labels.size
    n_pred = pred_labels.size
    # contingency matrix: rows pred, cols true
    cont = np.zeros((n_pred, n_true), dtype=int)
    for i, p in enumerate(pred_labels):
        for j, t in enumerate(true_labels):
            cont[i, j] = np.sum((y_pred == p) & (y_true == t))
    # maximize assignment -> minimize negative
    row_ind, col_ind = linear_sum_assignment(-cont)
    total = cont[row_ind, col_ind].sum()
    acc = total / y_true.shape[0]
    mapping = {pred_labels[r]: true_labels[c] for r, c in zip(row_ind, col_ind)}
    return acc, mapping

def compute_sse(X, labels, centers=None):
    X = np.asarray(X)
    labels = np.asarray(labels)
    unique = np.unique(labels)
    if centers is None:
        centers = []
        for lbl in unique:
            pts = X[labels == lbl]
            if pts.size == 0:
                centers.append(np.zeros(X.shape[1]))
            else:
                centers.append(pts.mean(axis=0))
        centers = np.vstack(centers)
    sse = 0.0
    for i, lbl in enumerate(unique):
        pts = X[labels == lbl]
        if pts.size == 0:
            continue
        c = centers[i]
        sse += ((pts - c) ** 2).sum()
    return float(sse)

def compute_ssb(X, labels):
    X = np.asarray(X)
    overall_mean = X.mean(axis=0)
    ssb = 0.0
    for lbl in np.unique(labels):
        pts = X[labels == lbl]
        n = pts.shape[0]
        if n == 0:
            continue
        mean = pts.mean(axis=0)
        ssb += n * ((mean - overall_mean) ** 2).sum()
    return float(ssb)



In [10]:
def compute_metrics(X, y_true, y_pred):
    metrics = {}
    # Pairwise / information
    metrics['rand_score'] = rand_score(y_true, y_pred)
    metrics['adjusted_rand_score'] = adjusted_rand_score(y_true, y_pred)
    metrics['mutual_info'] = mutual_info_score(y_true, y_pred)
    metrics['adjusted_mutual_info'] = adjusted_mutual_info_score(y_true, y_pred)
    metrics['normalized_mutual_info'] = normalized_mutual_info_score(y_true, y_pred)
    # Internal metrics (some require >1 cluster)
    try:
        metrics['silhouette'] = silhouette_score(X, y_pred)
    except Exception:
        metrics['silhouette'] = np.nan
    try:
        metrics['calinski_harabasz'] = calinski_harabasz_score(X, y_pred)
    except Exception:
        metrics['calinski_harabasz'] = np.nan
    try:
        metrics['davies_bouldin'] = davies_bouldin_score(X, y_pred)
    except Exception:
        metrics['davies_bouldin'] = np.nan
    # Accuracy via Hungarian mapping
    acc, mapping = best_cluster_accuracy(y_true, y_pred)
    metrics['accuracy'] = acc
    metrics['mapping'] = mapping
    # SSE & SSB (treat clusters including noise label if present)
    unique = np.unique(y_pred)
    centers = np.vstack([X[y_pred == lbl].mean(axis=0) if np.any(y_pred == lbl) else np.zeros(X.shape[1]) for lbl in unique])
    metrics['SSE'] = compute_sse(X, y_pred, centers=centers)
    metrics['SSB'] = compute_ssb(X, y_pred)
    return metrics

In [11]:
# PAM fallback (if KMedoids not present)

def pam(X, n_clusters=3, max_iter=300, random_state=None):
    X = np.asarray(X)
    rng = np.random.default_rng(random_state)
    n = X.shape[0]
    # initialize medoids randomly
    medoid_idxs = rng.choice(n, size=n_clusters, replace=False)
    D = cdist(X, X, metric='euclidean')
    labels = np.argmin(D[:, medoid_idxs], axis=1)
    for it in range(max_iter):
        changed = False
        for i in range(n_clusters):
            current_medoid = medoid_idxs[i]
            for cand in range(n):
                if cand in medoid_idxs:
                    continue
                new_medoids = medoid_idxs.copy()
                new_medoids[i] = cand
                new_cost = D[:, new_medoids].min(axis=1).sum()
                cur_cost = D[:, medoid_idxs].min(axis=1).sum()
                if new_cost < cur_cost - 1e-8:
                    medoid_idxs = new_medoids
                    changed = True
                    break
            if changed:
                break
        labels = np.argmin(D[:, medoid_idxs], axis=1)
        if not changed:
            break
    medoids = X[medoid_idxs]
    return medoid_idxs, medoids, labels

In [12]:
# Bisecting KMeans

from sklearn.cluster import KMeans as SKKMeans
class BisectingKMeans:
    def __init__(self, n_clusters=3, random_state=None, n_init=10, max_iter=300):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.n_init = n_init
        self.max_iter = max_iter

    def fit_predict(self, X):
        X = np.asarray(X)
        clusters = {0: np.arange(X.shape[0])}
        next_label = 1
        while len(clusters) < self.n_clusters:
            # choose cluster with largest SSE
            sse_map = {}
            for lbl, idxs in clusters.items():
                if idxs.size == 0:
                    sse_map[lbl] = 0.0
                else:
                    center = X[idxs].mean(axis=0)
                    sse_map[lbl] = ((X[idxs] - center) ** 2).sum()
            to_split = max(sse_map.items(), key=lambda x: x[1])[0]
            idxs = clusters.pop(to_split)
            if idxs.size <= 1:
                clusters[next_label] = np.array([], dtype=int)
                next_label += 1
                continue
            k2 = SKKMeans(n_clusters=2, random_state=self.random_state, n_init=self.n_init, max_iter=self.max_iter)
            sub_labels = k2.fit_predict(X[idxs])
            clusters[to_split] = idxs[sub_labels == 0]
            clusters[next_label] = idxs[sub_labels == 1]
            next_label += 1
            if next_label > X.shape[0] + 5:
                break
        labels_out = np.empty(X.shape[0], dtype=int)
        for lbl, idxs in clusters.items():
            labels_out[idxs] = lbl
        # remap to contiguous
        uniq = np.unique(labels_out)
        mapping = {old: new for new, old in enumerate(uniq)}
        labels_contig = np.array([mapping[l] for l in labels_out])
        return labels_contig

# Hierarchical helper

def hierarchical_try_linkages(X, n_clusters, linkages=('ward','average','complete','single')):
    results = {}
    for link in linkages:
        try:
            Z = linkage(X, method=link, metric='euclidean')
            labs = fcluster(Z, t=n_clusters, criterion='maxclust') - 1
            results[link] = (Z, labs)
        except Exception:
            continue
    return results

In [13]:
# Runner per dataset

def run_experiments_on_dataset(name, X, y, random_state=42, kmeans_n_init=20, best_of_n_kmeans=10, save_plots_prefix=None):
    print(f"\n==== Dataset: {name} | samples={X.shape[0]} features={X.shape[1]} classes={len(np.unique(y))} ====")
    y = relabel_classes(y)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    n_clusters = len(np.unique(y))

    records = []

    # 1) Partition-based: KMeans (random init) best-of-N

    print("Running KMeans (best-of-N random init)...")
    best_acc = -1; best_labels = None; best_model = None
    for i in range(best_of_n_kmeans):
        km = KMeans(n_clusters=n_clusters, init='random', n_init=1, random_state=random_state + i, max_iter=300)
        labs = km.fit_predict(Xs)
        acc, _ = best_cluster_accuracy(y, labs)
        if acc > best_acc:
            best_acc = acc; best_labels = labs; best_model = km
    km_metrics = compute_metrics(Xs, y, best_labels)
    records.append({
        'dataset': name, 'algorithm': f'KMeans (best-of-{best_of_n_kmeans} random)', 'n_clusters': n_clusters,
        'accuracy': km_metrics['accuracy'], 'rand_score': km_metrics['rand_score'],
        'adjusted_rand_score': km_metrics['adjusted_rand_score'],
        'mutual_info': km_metrics['mutual_info'], 'adjusted_mutual_info': km_metrics['adjusted_mutual_info'],
        'normalized_mutual_info': km_metrics['normalized_mutual_info'],
        'silhouette': km_metrics['silhouette'], 'calinski_harabasz': km_metrics['calinski_harabasz'],
        'davies_bouldin': km_metrics['davies_bouldin'], 'SSE': km_metrics['SSE'], 'SSB': km_metrics['SSB']
    })
    print(" KMeans best accuracy: {:.4f}".format(km_metrics['accuracy']))

    # 2) KMeans++ (scikit-learn) best-of-n_init variants

    print("Running KMeans++ (scikit-learn with multiple n_init)...")
    best_acc = -1; best_labels = None
    for i in range(best_of_n_kmeans):
        kmpp = KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=random_state + i)
        labs = kmpp.fit_predict(Xs)
        acc, _ = best_cluster_accuracy(y, labs)
        if acc > best_acc:
            best_acc = acc; best_labels = labs
    kmpp_metrics = compute_metrics(Xs, y, best_labels)
    records.append({
        'dataset': name, 'algorithm': f'KMeans++ (best-of-{best_of_n_kmeans})', 'n_clusters': n_clusters,
        'accuracy': kmpp_metrics['accuracy'], 'rand_score': kmpp_metrics['rand_score'],
        'adjusted_rand_score': kmpp_metrics['adjusted_rand_score'],
        'mutual_info': kmpp_metrics['mutual_info'], 'adjusted_mutual_info': kmpp_metrics['adjusted_mutual_info'],
        'normalized_mutual_info': kmpp_metrics['normalized_mutual_info'],
        'silhouette': kmpp_metrics['silhouette'], 'calinski_harabasz': kmpp_metrics['calinski_harabasz'],
        'davies_bouldin': kmpp_metrics['davies_bouldin'], 'SSE': kmpp_metrics['SSE'], 'SSB': kmpp_metrics['SSB']
    })
    print(" KMeans++ best accuracy: {:.4f}".format(kmpp_metrics['accuracy']))

    # 3) KMedoids / PAM

    print("Running K-Medoids / PAM...")
    if _has_kmedoids:
        kmdd = KMedoids(n_clusters=n_clusters, method='pam', random_state=random_state)
        labs = kmdd.fit_predict(Xs)
        centers = None
    else:
        medoid_idxs, medoids, labs = pam(Xs, n_clusters=n_clusters, random_state=random_state)
        centers = medoids
    kmed_metrics = compute_metrics(Xs, y, labs)
    records.append({
        'dataset': name, 'algorithm': 'KMedoids/PAM', 'n_clusters': n_clusters,
        'accuracy': kmed_metrics['accuracy'], 'rand_score': kmed_metrics['rand_score'],
        'adjusted_rand_score': kmed_metrics['adjusted_rand_score'],
        'mutual_info': kmed_metrics['mutual_info'], 'adjusted_mutual_info': kmed_metrics['adjusted_mutual_info'],
        'normalized_mutual_info': kmed_metrics['normalized_mutual_info'],
        'silhouette': kmed_metrics['silhouette'], 'calinski_harabasz': kmed_metrics['calinski_harabasz'],
        'davies_bouldin': kmed_metrics['davies_bouldin'], 'SSE': kmed_metrics['SSE'], 'SSB': kmed_metrics['SSB']
    })
    print(" KMedoids accuracy: {:.4f}".format(kmed_metrics['accuracy']))

    # 4) Bisecting KMeans

    print("Running Bisecting KMeans...")
    bkm = BisectingKMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labs = bkm.fit_predict(Xs)
    bkm_metrics = compute_metrics(Xs, y, labs)
    records.append({
        'dataset': name, 'algorithm': 'Bisecting KMeans', 'n_clusters': n_clusters,
        'accuracy': bkm_metrics['accuracy'], 'rand_score': bkm_metrics['rand_score'],
        'adjusted_rand_score': bkm_metrics['adjusted_rand_score'],
        'mutual_info': bkm_metrics['mutual_info'], 'adjusted_mutual_info': bkm_metrics['adjusted_mutual_info'],
        'normalized_mutual_info': bkm_metrics['normalized_mutual_info'],
        'silhouette': bkm_metrics['silhouette'], 'calinski_harabasz': bkm_metrics['calinski_harabasz'],
        'davies_bouldin': bkm_metrics['davies_bouldin'], 'SSE': bkm_metrics['SSE'], 'SSB': bkm_metrics['SSB']
    })
    print(" Bisecting KMeans accuracy: {:.4f}".format(bkm_metrics['accuracy']))

    # 5) Hierarchical (dendrogram): try multiple linkages, pick best by accuracy

    print("Running Hierarchical clustering (linkages: ward, average, complete, single)...")
    h_results = hierarchical_try_linkages(Xs, n_clusters, linkages=('ward','average','complete','single'))
    best_link = None; best_acc = -1; best_labs = None; best_Z=None
    for link, (Z, labs) in h_results.items():
        acc, _ = best_cluster_accuracy(y, labs)
        print(f"  linkage={link:8s} acc={acc:.4f}")
        if acc > best_acc:
            best_acc = acc; best_link = link; best_labs = labs; best_Z = Z
    if best_labs is not None:
        hier_metrics = compute_metrics(Xs, y, best_labs)
        records.append({
            'dataset': name, 'algorithm': f'Hierarchical ({best_link})', 'n_clusters': n_clusters,
            'accuracy': hier_metrics['accuracy'], 'rand_score': hier_metrics['rand_score'],
            'adjusted_rand_score': hier_metrics['adjusted_rand_score'],
            'mutual_info': hier_metrics['mutual_info'], 'adjusted_mutual_info': hier_metrics['adjusted_mutual_info'],
            'normalized_mutual_info': hier_metrics['normalized_mutual_info'],
            'silhouette': hier_metrics['silhouette'], 'calinski_harabasz': hier_metrics['calinski_harabasz'],
            'davies_bouldin': hier_metrics['davies_bouldin'], 'SSE': hier_metrics['SSE'], 'SSB': hier_metrics['SSB']
        })
        print(" Best hierarchical linkage: {} acc={:.4f}".format(best_link, best_acc))

        # Save dendrogram

        if save_plots_prefix and best_Z is not None:
            plt.figure(figsize=(10, 5))
            dendrogram(best_Z, truncate_mode='level', p=30)
            plt.title(f"{name} dendrogram (linkage={best_link})")
            plt.xlabel("Samples (truncated)")
            plt.ylabel("Distance")
            out = f"{save_plots_prefix}_{name}_dendrogram_{best_link}.png"
            plt.tight_layout()
            plt.savefig(out, dpi=150)
            plt.close()
            print("  Saved dendrogram to:", out)
    else:
        print("  No hierarchical clustering succeeded.")

    # 6) Density-based: DBSCAN

    print("Running DBSCAN (default eps/min_samples heuristics)...")

    # choose eps heuristically: use 0.5 as baseline; user can tweak

    db = DBSCAN(eps=0.5, min_samples=5, metric='euclidean')
    labs = db.fit_predict(Xs)  # -1 is noise
    db_metrics = compute_metrics(Xs, y, labs)
    records.append({
        'dataset': name, 'algorithm': 'DBSCAN (eps=0.5,min_samples=5)', 'n_clusters': len(np.unique(labs)),
        'accuracy': db_metrics['accuracy'], 'rand_score': db_metrics['rand_score'],
        'adjusted_rand_score': db_metrics['adjusted_rand_score'],
        'mutual_info': db_metrics['mutual_info'], 'adjusted_mutual_info': db_metrics['adjusted_mutual_info'],
        'normalized_mutual_info': db_metrics['normalized_mutual_info'],
        'silhouette': db_metrics['silhouette'], 'calinski_harabasz': db_metrics['calinski_harabasz'],
        'davies_bouldin': db_metrics['davies_bouldin'], 'SSE': db_metrics['SSE'], 'SSB': db_metrics['SSB']
    })
    print(" DBSCAN clusters (including noise=-1):", np.unique(labs))

    # 7) Density-based: OPTICS

    print("Running OPTICS...")
    opt = OPTICS(min_samples=5, metric='euclidean')
    labs = opt.fit_predict(Xs)  # -1 is noise
    opt_metrics = compute_metrics(Xs, y, labs)
    records.append({
        'dataset': name, 'algorithm': 'OPTICS (min_samples=5)', 'n_clusters': len(np.unique(labs)),
        'accuracy': opt_metrics['accuracy'], 'rand_score': opt_metrics['rand_score'],
        'adjusted_rand_score': opt_metrics['adjusted_rand_score'],
        'mutual_info': opt_metrics['mutual_info'], 'adjusted_mutual_info': opt_metrics['adjusted_mutual_info'],
        'normalized_mutual_info': opt_metrics['normalized_mutual_info'],
        'silhouette': opt_metrics['silhouette'], 'calinski_harabasz': opt_metrics['calinski_harabasz'],
        'davies_bouldin': opt_metrics['davies_bouldin'], 'SSE': opt_metrics['SSE'], 'SSB': opt_metrics['SSB']
    })
    print(" OPTICS clusters (including noise=-1):", np.unique(labs))

    df = pd.DataFrame(records)

    # Reorder columns

    cols = ['dataset','algorithm','n_clusters','accuracy','rand_score','adjusted_rand_score',
            'mutual_info','adjusted_mutual_info','normalized_mutual_info',
            'silhouette','calinski_harabasz','davies_bouldin','SSE','SSB']
    df = df[cols]
    return df

In [14]:
# Main

def main():
    iris = datasets.load_iris()
    wine = datasets.load_wine()

    df_iris = run_experiments_on_dataset("Iris", iris.data, iris.target, random_state=42, save_plots_prefix="plot")
    df_wine = run_experiments_on_dataset("Wine", wine.data, wine.target, random_state=42, save_plots_prefix="plot")

    print("\n\n=== Iris Results ===")
    pd.set_option('display.float_format', lambda x: f'{x:0.4f}')
    print(df_iris.to_string(index=False))

    print("\n\n=== Wine Results ===")
    print(df_wine.to_string(index=False))

    df_all = pd.concat([df_iris, df_wine], ignore_index=True)
    df_all.to_csv("clustering_full_results.csv", index=False)
    print("\nSaved combined results to clustering_full_results.csv")

    # Report best algorithm per dataset (by accuracy)
    for name, df in [("Iris", df_iris), ("Wine", df_wine)]:
        best = df.loc[df['accuracy'].idxmax()]
        print(f"\n{name}: best algorithm = {best['algorithm']} (accuracy={best['accuracy']:.4f})")
        if best['accuracy'] >= 0.80:
            print("  -> Achieved accuracy >= 80%")
        else:
            print("     -> Did NOT reach 80% accuracy. Suggestions:")
            print("     - Try increasing KMeans n_init or best-of-N runs.")
            print("     - For Wine, try PCA to reduce dimensionality before clustering (often helps).")
            print("     - For DBSCAN/OPTICS tune eps / min_samples and examine reachability plot for OPTICS.")
    print("\n")

if __name__ == "__main__":
    main()


==== Dataset: Iris | samples=150 features=4 classes=3 ====
Running KMeans (best-of-N random init)...
 KMeans best accuracy: 0.8333
Running KMeans++ (scikit-learn with multiple n_init)...
 KMeans++ best accuracy: 0.8333
Running K-Medoids / PAM...
 KMedoids accuracy: 0.8467
Running Bisecting KMeans...
 Bisecting KMeans accuracy: 0.8333
Running Hierarchical clustering (linkages: ward, average, complete, single)...
  linkage=ward     acc=0.8267
  linkage=average  acc=0.6867
  linkage=complete acc=0.7867
  linkage=single   acc=0.6600
 Best hierarchical linkage: ward acc=0.8267
  Saved dendrogram to: plot_Iris_dendrogram_ward.png
Running DBSCAN (default eps/min_samples heuristics)...
 DBSCAN clusters (including noise=-1): [-1  0  1]
Running OPTICS...
 OPTICS clusters (including noise=-1): [-1  0  1  2  3  4]

==== Dataset: Wine | samples=178 features=13 classes=3 ====
Running KMeans (best-of-N random init)...
 KMeans best accuracy: 0.9719
Running KMeans++ (scikit-learn with multiple n_init)